# Descargar un slice de la data de inducción BARC (para de-risk del SFT)

`barc0/induction_heavy_100k_jsonl` (HF, formato chat `messages`). Kernel T4+internet → guarda
un slice a `/kaggle/working` → se adjunta al kernel de SFT (L4, offline) como kernel_source.

In [ ]:
!pip -q install -U datasets >/dev/null 2>&1
import json, time
from datasets import load_dataset
N = 6000  # slice para validar la infra de SFT (no todo el 100k)
t0 = time.time()
ds = load_dataset('barc0/induction_heavy_100k_jsonl', split='train_sft', streaming=True)
rows = []
for i, r in enumerate(ds):
    rows.append(r)
    if len(rows) >= N:
        break
print('bajados', len(rows), 'ejemplos en', round(time.time()-t0), 's')

In [ ]:
import os
os.makedirs('/kaggle/working/barc_induction_slice', exist_ok=True)
out = '/kaggle/working/barc_induction_slice/induction_heavy_slice.jsonl'
with open(out, 'w') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')
sz = os.path.getsize(out) / 1024**2
print(f'escrito {out}: {len(rows)} ejemplos, {sz:.1f} MB')
# sanity: estructura del primer ejemplo
m = rows[0]['messages']
print('roles:', [x['role'] for x in m])
assert [x['role'] for x in m] == ['system', 'user', 'assistant'], 'formato inesperado'
print('OK: slice de data listo en el output del kernel')